In [1]:
import os
import sys
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")
github_user = "LeonardoLeoo89"
repo_name = "pokemon-predictor"
repo_path = f"/kaggle/working/{repo_name}"

!rm -rf {repo_path}
!git clone https://{github_token}@github.com/{github_user}/{repo_name}.git

print("\n--- Directory Check ---")
if os.path.exists(repo_path):
    print("Files found in repo:", os.listdir(repo_path))
else:
    print(f"Error: {repo_path} was not created. Check your git clone output above.")

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from src import *

TRAIN_PATH = "/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/train.jsonl"
TEST_PATH = "/kaggle/input/competitions/fds-pokemon-battles-prediction-2026/test.jsonl"

train_battles = load_jsonl(TRAIN_PATH)

print(f"Loaded {len(train_battles)} training battles.")


Cloning into 'pokemon-predictor'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 70 (delta 36), reused 46 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 48.70 KiB | 4.06 MiB/s, done.
Resolving deltas: 100% (36/36), done.

--- Directory Check ---
Files found in repo: ['README.md', 'LICENSE', '.git', 'src', '.idea', 'requirements.txt']
Loaded 10000 training battles.


In [2]:
import pandas as pd

pipeline = PokemonBattlePredictionPipeline(
        min_correlation=0.018,
        n_splits_cv=10,
        random_state=42,
)
pipeline.fit(train_battles, verbose=True)

# Display 10-Fold CV Performance Comparison (Table 1 of report)
display(pipeline.cv_summary_table())
print(f"Predicting on test data...")
test_records = load_jsonl(TEST_PATH)
probs = pipeline.predict_proba(test_records)[:, 1]
preds = pipeline.predict(test_records)
out_df = pd.DataFrame({
    "battle_id": [b.get("battle_id", f"test_{i}") for i, b in enumerate(test_records)],
    "player_won": preds,
})
out_df.to_csv("submission.csv", index=False)
print("\n'submission.csv' file created successfully!")
!rm -rf {repo_path}
display(out_df.head())

Fitting pipeline on 10000 battle records...
  Feature extraction complete in 47.02s (200 features).
  Retained 140 high-signal features (pruned 60 noisy/constant features).
  Executing 10-Fold CV to calibrate ensemble weights and threshold...
  Optimal SLSQP Simplex Weights:
    - hist_gbdt_1           : 0.2885 (28.8%)
    - hist_gbdt_2           : 0.2482 (24.8%)
    - logistic_regression   : 0.4631 (46.3%)
    - pls_soft_model        : 0.0000 (0.0%)
    - extra_trees           : 0.0002 (0.0%)
    - xgboost               : 0.0000 (0.0%)
  Calibrated Power Exponent: r = 0.90
  Calibrated Decision Threshold: t = 0.503

     OUT-OF-FOLD CROSS-VALIDATION STATISTICAL DATA (10-FOLD)     
  Metric                    | Value          
-----------------------------------------------------------------
  Accuracy                  |    69.57%
  ROC-AUC                   |    76.74%
  Brier Score Loss          |    0.1958
  F1-Score                  |    69.74%
  Precision                 |    69.3

,battle_id,player_won
0,0,1
1,1,0
2,2,0
3,3,0
4,4,0
